# 03b — Embeddings + ChromaDB

## Mục tiêu
- Biến mỗi chunk thành vector (embedding).
- Lưu vào Chroma local và thử retrieve top-k đoạn liên quan.

## Kiến thức cần nhớ
- Embedding: câu gần nghĩa → vector gần nhau (dù khác từ).
- Vector DB giúp tìm đoạn liên quan nhanh hơn so khớp chuỗi thô.
- Cần `OPENAI_API_KEY` trong file `.env` ở gốc repo (xem `.env.example`).

**Không** hardcode API key trong notebook.


## 1. Load API key từ .env


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path("../../").resolve()
load_dotenv(ROOT / ".env")

key = os.getenv("OPENAI_API_KEY", "")
if not key or key.startswith("sk-your"):
    raise EnvironmentError(
        "Chua co OPENAI_API_KEY hop le. Copy .env.example -> .env va dien key."
    )
print("API key: da load (do dai", len(key), "ky tu)")


## 2. Load + split lại (giống 03a)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []
for pdf in sorted((ROOT / "docs").glob("*.pdf")):
    docs.extend(PyPDFLoader(str(pdf)).load())

chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
print("chunks:", len(chunks))


## 3. Tạo Chroma từ embeddings


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

persist_dir = str(ROOT / "data" / "chroma_db")
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Da luu Chroma tai:", persist_dir)


## 4. Thử retrieve (chưa gọi LLM trả lời)


In [ ]:
query = "Chatbot RAG dung cong cu gi?"
hits = retriever.invoke(query)
for i, doc in enumerate(hits, 1):
    src = doc.metadata.get("source", "?")
    print(f"\n=== Hit {i} | {Path(src).name} ===")
    print(doc.page_content[:350])


## Checklist
- [ ] Key chỉ nằm trong `.env`
- [ ] Hiểu embedding khác TF-IDF chỗ nào (ngữ nghĩa vs đếm từ)
- [ ] Retrieve trả về chunk liên quan trước khi có LLM
